# CAMeL-BERT Inference Pipeline (GPU/Colab)

**Purpose**: Run inference on full Kitab Uqala and export results for local analysis

**Output**: JSON with token predictions + offsets (ready for local post-processing)

**No analysis here** — just inference and export

In [3]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"[OK] Working directory: {os.getcwd()}")

Mounted at /content/drive
[OK] Working directory: /content/drive/MyDrive/khabar-segmentation


In [2]:
# Install dependencies
!pip install transformers torch tqdm -q
print("[OK] Dependencies installed")

[OK] Dependencies installed


In [1]:
# Imports
import json
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU: Tesla T4


In [4]:
# Load model and tokenizer
model_path = Path('checkpoints/camelbert_binary_classification_final')

print(f"[INFO] Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()

if torch.cuda.is_available():
    model = model.cuda()

print(f"[OK] Model loaded and ready")

[INFO] Loading model from checkpoints/camelbert_binary_classification_final...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[OK] Model loaded and ready


In [6]:
# Load the Kitab Uqala corpus
corpus_file = Path('data/processed/kitab_uqala_reference_corpus.txt')

print(f"[INFO] Loading corpus from {corpus_file}...")
with open(corpus_file, encoding='utf-8') as f:
    full_text = f.read()

print(f"[OK] Corpus loaded")
print(f"  Size: {len(full_text):,} chars")
print(f"  Words: {len(full_text.split()):,}")

[INFO] Loading corpus from data/processed/kitab_uqala_reference_corpus.txt...
[OK] Corpus loaded
  Size: 268,540 chars
  Words: 53,812


In [10]:
# ============================================================================
# STEP 1: Run inference on full text with offset mapping
# ============================================================================
def infer_with_offsets(text: str, tokenizer, model, chunk_size: int = 512, overlap: int = 50) -> dict:
    """
    Run inference on FULL document in overlapping chunks.

    Process in chunks to handle documents larger than max_length.
    Chunks overlap to catch boundaries at chunk edges.
    """
    print(f"[INFO] Processing {len(text):,} chars in chunks of {chunk_size} with {overlap} overlap...\n")

    all_predictions = []
    all_probabilities = []
    all_offsets = []
    all_tokens = []

    chunk_num = 0
    pos = 0

    while pos < len(text):
        chunk_num += 1
        chunk_start = pos
        chunk_end = min(pos + chunk_size, len(text))
        chunk = text[chunk_start:chunk_end]

        print(f"  Chunk {chunk_num}: chars {chunk_start:,}-{chunk_end:,} ({len(chunk)} chars)")

        # Tokenize this chunk
        encoded = tokenizer(
            chunk,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_offsets_mapping=True,
            return_tensors='pt'
        )

        # Run inference
        with torch.no_grad():
            if torch.cuda.is_available():
                input_ids = encoded['input_ids'].cuda()
                attention_mask = encoded['attention_mask'].cuda()
                outputs = model(input_ids, attention_mask=attention_mask)
            else:
                outputs = model(**encoded)

            logits = outputs.logits[0]  # [seq_len, 2]

        # Get predictions and probabilities
        preds = np.argmax(logits.cpu().numpy(), axis=-1)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[:, 1]

        tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
        offsets = encoded['offset_mapping'][0].numpy()

        # Adjust offsets by chunk position
        adjusted_offsets = []
        for token_start, token_end in offsets:
            # Convert local offset to global document position
            global_start = chunk_start + token_start
            global_end = chunk_start + token_end
            adjusted_offsets.append((global_start, global_end))

        # Append to global lists
        all_predictions.extend(preds)
        all_probabilities.extend(probs)
        all_offsets.extend(adjusted_offsets)
        all_tokens.extend(tokens)

        # Move to next chunk
        # Non-overlapping: move by full chunk_size
        # With overlap: move by (chunk_size - overlap)
        pos += (chunk_size - overlap)

        if pos >= len(text):
            break

    print(f"[OK] Processed {chunk_num} chunks")
    print(f"[OK] Total tokens: {len(all_predictions)}")

    return {
        'predictions': all_predictions,
        'probabilities': all_probabilities,
        'tokens': all_tokens,
        'offsets': all_offsets,
    }

print("[OK] Inference function defined")

[OK] Inference function defined


In [11]:
# Run inference
print(f"[INFO] Running inference on full corpus...")
print(f"  Text length: {len(full_text):,} chars")
print(f"  Expected tokens: ~{len(full_text) // 4} (rough estimate)")
print(f"  Processing in 512-token chunks...\n")

inference_result = infer_with_offsets(full_text, tokenizer, model)

print(f"[OK] Inference complete")
print(f"  Total tokens: {len(inference_result['tokens'])}")
print(f"  Boundary tokens: {sum(inference_result['predictions'])}")
print(f"  Boundary ratio: {sum(inference_result['predictions']) / len(inference_result['predictions']) * 100:.1f}%")

[INFO] Running inference on full corpus...
  Text length: 268,540 chars
  Expected tokens: ~67135 (rough estimate)
  Processing in 512-token chunks...

[INFO] Processing 268,540 chars in chunks of 512 with 50 overlap...

  Chunk 1: chars 0-512 (512 chars)
  Chunk 2: chars 462-974 (512 chars)
  Chunk 3: chars 924-1,436 (512 chars)
  Chunk 4: chars 1,386-1,898 (512 chars)
  Chunk 5: chars 1,848-2,360 (512 chars)
  Chunk 6: chars 2,310-2,822 (512 chars)
  Chunk 7: chars 2,772-3,284 (512 chars)
  Chunk 8: chars 3,234-3,746 (512 chars)
  Chunk 9: chars 3,696-4,208 (512 chars)
  Chunk 10: chars 4,158-4,670 (512 chars)
  Chunk 11: chars 4,620-5,132 (512 chars)
  Chunk 12: chars 5,082-5,594 (512 chars)
  Chunk 13: chars 5,544-6,056 (512 chars)
  Chunk 14: chars 6,006-6,518 (512 chars)
  Chunk 15: chars 6,468-6,980 (512 chars)
  Chunk 16: chars 6,930-7,442 (512 chars)
  Chunk 17: chars 7,392-7,904 (512 chars)
  Chunk 18: chars 7,854-8,366 (512 chars)
  Chunk 19: chars 8,316-8,828 (512 chars)
  

In [13]:
# ============================================================================
# STEP 2: Export results to JSON
# ============================================================================

# Convert numpy types to Python types for JSON serialization
predictions_list = [int(p) for p in inference_result['predictions']]
probabilities_list = [float(p) for p in inference_result['probabilities']]
offsets_list = [(int(s), int(e)) for s, e in inference_result['offsets']]

# Export results
export_data = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus',
        'text_size_chars': len(full_text),
        'text_size_words': len(full_text.split()),
        'model': 'camelbert_binary_classification_final',
        'fix_applied': 'Full document chunking with overlap'
    },
    'inference_results': {
        'total_tokens': len(predictions_list),
        'boundary_tokens': sum(predictions_list),
        'predictions': predictions_list,
        'probabilities': probabilities_list,
        'tokens': inference_result['tokens'],
        'offsets': offsets_list,
    }
}

print(f"[OK] Data converted to JSON-serializable format")

# Save to Drive
output_file = Path('results/camelbert_kitab_uqala_raw_inference.json')
output_file.parent.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Saving inference results to {output_file}...")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

# Check file size
file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"[OK] File saved: {file_size_mb:.1f} MB")
print(f"[OK] Location on Drive: results/camelbert_kitab_uqala_raw_inference.json")

[OK] Data converted to JSON-serializable format
[INFO] Saving inference results to results/camelbert_kitab_uqala_raw_inference.json...
[OK] File saved: 28.5 MB
[OK] Location on Drive: results/camelbert_kitab_uqala_raw_inference.json


---

## Download Instructions

1. In Colab Files panel (left sidebar) → click Refresh
2. Navigate to `results/camelbert_kitab_uqala_raw_inference.json`
3. Right-click → Download
4. Save to your local `results/` directory

---

## What's in the file

- `predictions`: Binary token predictions (0/1 array, length 512)
- `probabilities`: Confidence scores for each token
- `tokens`: Token strings (for debugging)
- `offsets`: Character-level positions for each token **← Critical for local processing**

This is everything you need locally to:
1. Cluster boundary tokens
2. Extract actual segments
3. Compare with gold standard
4. Calculate metrics

In [ ]:
# Optional: Quick stats on predictions
print("\n[SUMMARY]")
print(f"Inference complete! Ready for local post-processing.\n")
print(f"Raw token predictions:")
print(f"  Total tokens: {export_data['inference_results']['total_tokens']}")
print(f"  Boundary tokens: {export_data['inference_results']['boundary_tokens']}")
print(f"  Mean boundary prob: {np.mean([p for p, pred in zip(inference_result['probabilities'], inference_result['predictions']) if pred == 1]):.4f}")
print(f"\nNext: Download the JSON file and run local post-processing")

## Local Processing (After Download)

```bash
python3 scripts/camelbert_local_postprocess.py \
  --input results/camelbert_kitab_uqala_raw_inference.json \
  --output results/camelbert_kitab_uqala_segments.json
```

This will:
1. Cluster boundary tokens
2. Extract text segments
3. Compare with 613 khabar gold standard
4. Calculate recall/precision/F1